# Course Correct Labs Reasoning Stability Observatory

**Unified Analysis System for CCL Empirical Studies**

This notebook provides a comprehensive analysis of four CCL studies:
1. **Mirror Loop** - Information collapse in iterative self-critique
2. **Recursive Confabulation** - Fabrication persistence across turns
3. **Violation State** - Contamination from refusal states
4. **Echo Chamber** - Belief percolation in multi-agent systems

**Runtime:** <10 minutes using existing data only

**Budget:** $0 (uses precomputed data; optional live demo <$0.05)

## Setup

In [ ]:
# Standard imports
import sys
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# CCL Observatory imports
from course_correct_evals import (
    MirrorLoopImporter,
    ConfabulationImporter,
    ViolationStateImporter,
    EchoChamberImporter,
    CrossStudyAnalysis,
)

from course_correct_evals.analysis.viz import (
    plot_four_panel_comparison,
    plot_leaderboard,
    plot_mirror_loop_detail,
)

from course_correct_evals.reports import export_csv_results, export_pdf_report

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Plot settings
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

# Suppress warnings for clean output
warnings.filterwarnings('ignore')

print("✓ Setup complete")

## Installation (Run this first in Colab!)

**If running in Google Colab**, run the cell below to install the package.  
**If running locally**, make sure you've run `pip install -e .` from the repository root.

In [ ]:
# Standard imports
import sys
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set paths
sys.path.insert(0, '..')

# CCL Observatory imports
from course_correct_evals import (
    MirrorLoopImporter,
    ConfabulationImporter,
    ViolationStateImporter,
    EchoChamberImporter,
    CrossStudyAnalysis,
)

from course_correct_evals.analysis.viz import (
    plot_four_panel_comparison,
    plot_leaderboard,
    plot_mirror_loop_detail,
)

from course_correct_evals.reports import export_csv_results, export_pdf_report

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Plot settings
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

# Suppress warnings for clean output
warnings.filterwarnings('ignore')

print("✓ Setup complete")

## 1. Load All Studies

The Observatory will attempt to load data from all four studies.
Studies without available data will be skipped gracefully.

In [ ]:
# Initialize Observatory
observatory = CrossStudyAnalysis()

# Load all available studies
# Data paths can be specified explicitly or auto-discovered
loaded_studies = observatory.load_all_studies(
    fail_on_missing=False  # Continue even if some studies missing
)

print("\nStudy Loading Status:")
for study, loaded in loaded_studies.items():
    status = "✓ Loaded" if loaded else "✗ Not Available"
    print(f"  {study}: {status}")

## 2. Cross-Study Leaderboard

Compare model performance across all four studies.

In [ ]:
# Generate leaderboard
leaderboard = observatory.create_leaderboard()

print("\nCross-Study Model Leaderboard:")
print("=" * 80)
display(leaderboard)

# Visualize leaderboard
if len(leaderboard) > 0:
    plot_leaderboard(leaderboard)

# Generate flagship figure
fig = plot_four_panel_comparison(
    observatory,
    figsize=(16, 12),
    save_path='four_panel_comparison.png'
)

plt.show()

## 3. Four-Panel Comparison Figure

Publication-quality visualization comparing all four studies.

In [ ]:
# Generate flagship figure
fig = plot_four_panel_comparison(
    observatory,
    figsize=(16, 12),
    save_path='../results/four_panel_comparison.png'
)

plt.show()

## 4. Mirror Loop Deep Dive

Detailed analysis of information collapse in iterative self-critique.

In [ ]:
if observatory._data_loaded['mirror_loop']:
    # Run analysis
    ml_analysis = observatory.analyze_mirror_loop()
    
    print("\nMirror Loop Study Results:")
    print("=" * 80)
    print(f"Total Sequences: {ml_analysis['total_sequences']}")
    print(f"Collapsed Sequences: {ml_analysis['collapsed_sequences']}")
    print(f"Collapse Rate: {ml_analysis['collapse_rate']:.1%}")
    print(f"Mean ΔI: {ml_analysis['mean_delta_i_overall']:.3f}")
    
    # Model-level statistics
    if ml_analysis['model_statistics'] is not None:
        print("\nPer-Model Statistics:")
        display(ml_analysis['model_statistics'])
    
    # Sequence analysis sample
    print("\nSample Sequence Analysis:")
    display(ml_analysis['sequence_analysis'].head(10))
    
    # Detailed plot for one sequence
    sequence_ids = observatory.mirror_loop_data['sequence_id'].unique()
    if len(sequence_ids) > 0:
        plot_mirror_loop_detail(observatory, sequence_id=sequence_ids[0])
        plt.show()
else:
    print("Mirror Loop data not available")

## 5. Recursive Confabulation Deep Dive

Analysis of fabrication persistence and intervention effectiveness.

In [ ]:
if observatory._data_loaded['confabulation']:
    # Run analysis
    conf_analysis = observatory.analyze_confabulation()
    
    print("\nRecursive Confabulation Study Results:")
    print("=" * 80)
    print(f"Total Conversations: {conf_analysis['total_conversations']}")
    print(f"Total Turns: {conf_analysis['total_turns']}")
    
    # Persistence statistics
    pers_overall = conf_analysis['persistence_statistics']['overall']
    print("\nOverall Persistence:")
    print(f"  Total Fabrications: {pers_overall['total_fabrications']}")
    print(f"  Persistent: {pers_overall['persistent_fabrications']}")
    print(f"  Persistence Rate: {pers_overall['persistence_rate']:.1%}")
    
    # By intervention arm
    if 'by_intervention' in conf_analysis['persistence_statistics']:
        print("\nPersistence by Intervention Arm:")
        for arm, stats in conf_analysis['persistence_statistics']['by_intervention'].items():
            print(f"  {arm}:")
            print(f"    Fabrications: {stats['total_fabrications']}")
            print(f"    Persistence Rate: {stats['persistence_rate']:.1%}")
    
    # Intervention effectiveness
    if conf_analysis['intervention_effectiveness'] is not None:
        print("\nIntervention Effectiveness:")
        display(conf_analysis['intervention_effectiveness'])
else:
    print("Confabulation data not available")

## 6. Violation State Deep Dive

Analysis of contamination from violation requests and refusals.

In [ ]:
if observatory._data_loaded['violation_state']:
    # Run analysis
    vs_analysis = observatory.analyze_violation_state()
    
    print("\nViolation State Study Results:")
    print("=" * 80)
    print(f"Total Conversations: {vs_analysis['total_conversations']}")
    print(f"Total Turns: {vs_analysis['total_turns']}")
    
    # Contamination statistics
    contam_stats = vs_analysis['contamination_statistics']
    print("\nContamination Statistics:")
    print(f"  Contaminated Conversations: {contam_stats['contaminated_conversations']}")
    print(f"  Contaminated Turns: {contam_stats['contaminated_turns']}")
    print(f"  Contamination Rate: {contam_stats['contamination_rate']:.1%}")
    print(f"  Conversation Contamination Rate: {contam_stats['conversation_contamination_rate']:.1%}")
    
    # Refusal statistics
    print("\nRefusal Statistics:")
    display(vs_analysis['refusal_statistics'])
    
    # Response type distribution
    if 'response_type' in observatory.violation_state_data.columns:
        print("\nResponse Type Distribution:")
        response_dist = observatory.violation_state_data['response_type'].value_counts()
        display(response_dist)
else:
    print("Violation State data not available")

## 7. Echo Chamber Deep Dive

Analysis of belief percolation and radicalization in multi-agent systems.

In [ ]:
if observatory._data_loaded['echo_chamber']:
    # Run analysis
    echo_analysis = observatory.analyze_echo_chamber()
    
    print("\nEcho Chamber Study Results:")
    print("=" * 80)
    print(f"Total Simulations: {echo_analysis['total_simulations']}")
    print(f"Total Steps: {echo_analysis['total_steps']}")
    
    # Echo metrics
    echo_stats = echo_analysis['echo_statistics']
    print("\nEcho Chamber Metrics:")
    for metric_name, metric_stats in echo_stats['metrics'].items():
        print(f"\n{metric_name}:")
        print(f"  Mean: {metric_stats['mean']:.3f}")
        print(f"  Std: {metric_stats['std']:.3f}")
        print(f"  Range: [{metric_stats['min']:.3f}, {metric_stats['max']:.3f}]")
        if 'trend_mean' in metric_stats:
            print(f"  Mean Trend: {metric_stats['trend_mean']:.3f}")
            print(f"  Increasing Trajectories: {metric_stats['increasing_count']}")
            print(f"  Decreasing Trajectories: {metric_stats['decreasing_count']}")
    
    # Convergence statistics
    print("\nConvergence Statistics:")
    conv_stats = echo_analysis['convergence_statistics']
    for key, value in conv_stats.items():
        print(f"  {key}: {value}")
    
    # Trajectory sample
    print("\nSample Trajectories:")
    display(echo_analysis['trajectories'].head(10))
    
    # Threshold crossings
    print("\nThreshold Crossings:")
    for metric, crossings in echo_analysis['threshold_crossings'].items():
        if len(crossings) > 0:
            print(f"\n{metric}:")
            display(crossings.head())
else:
    print("Echo Chamber data not available")

## 8. Optional: Live Demo

**⚠️ WARNING: This cell costs money and requires API keys!**

To enable:
1. Set `RUN_LIVE_DEMO = True` below
2. Provide your API key
3. Run the cell

Estimated cost: $0.01-0.05 for 10 iterations

In [ ]:
# Export CSV results
print("Exporting results to CSV...\n")
exported_files = export_csv_results(observatory, output_dir='results')

print("\nExported Files:")
for result_type, file_path in exported_files.items():
    print(f"  {result_type}: {file_path}")

# Generate report
print("Generating report...\n")
report_path = export_pdf_report(
    observatory,
    output_path='ccl_observatory_report.pdf'
)

print(f"\nReport generated: {report_path}")

In [ ]:
## Summary

This notebook has provided a comprehensive analysis of the CCL Reasoning Stability Observatory.

### Key Findings

The Observatory synthesizes data across four empirical studies to identify:

1. **Information Collapse** - Models that exhibit rapid ΔI decay in self-critique
2. **Fabrication Persistence** - Models prone to maintaining false information
3. **State Contamination** - Models whose refusal behavior leaks across contexts
4. **Echo Chamber Effects** - Multi-agent systems prone to radicalization

### Next Steps

1. Review exported CSV results in `results/` folder (or download from Colab)
2. Examine the four-panel comparison figure
3. Use the leaderboard to compare model stability
4. Optionally: Run live demos with your own prompts

---

**Course Correct Labs**  
*Reasoning Stability Observatory v0.1.0*

In [ ]:
# Generate report
print("Generating report...\n")
report_path = export_pdf_report(
    observatory,
    output_path='../results/ccl_observatory_report.pdf'
)

print(f"\nReport generated: {report_path}")

## Summary

This notebook has provided a comprehensive analysis of the CCL Reasoning Stability Observatory.

### Key Findings

The Observatory synthesizes data across four empirical studies to identify:

1. **Information Collapse** - Models that exhibit rapid ΔI decay in self-critique
2. **Fabrication Persistence** - Models prone to maintaining false information
3. **State Contamination** - Models whose refusal behavior leaks across contexts
4. **Echo Chamber Effects** - Multi-agent systems prone to radicalization

### Next Steps

1. Review exported CSV results in `../results/`
2. Examine the four-panel comparison figure
3. Use the leaderboard to compare model stability
4. Optionally: Run live demos with your own prompts

---

**Course Correct Labs**  
*Reasoning Stability Observatory v0.1.0*